In [1]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces
from collections import deque
import random
import pygame

In [2]:
class SnakeEnv(gym.Env):
    """
    Entorno personalizado de Snake con obstáculos dinámicos y comida mala.
    Formulado como MDP episódico y parcialmente estocástico.
 
    Espacio de estados : vector de 14 elementos binarios/numéricos
    Espacio de acciones: Discrete(3)  ->  0=izquierda, 1=recto, 2=derecha
    """
 
    metadata = {"render_modes": ["human", "rgb_array"], "render_fps": 10}
 
    # Direcciones: (delta_fila, delta_columna)
    DIRECTIONS = [
        (-1,  0),   # 0: Arriba
        ( 0,  1),   # 1: Derecha
        ( 1,  0),   # 2: Abajo
        ( 0, -1),   # 3: Izquierda
    ]
 
    def __init__(self, grid_size=10, obstacle_move_freq=5,
                 max_steps=500, render_mode=None, cell_size=50):
        super().__init__()
 
        self.grid_size          = grid_size
        self.obstacle_move_freq = obstacle_move_freq
        self.max_steps          = max_steps
        self.render_mode        = render_mode
        self.cell_size          = cell_size
        self.n_obstacles        = 3
 
        # --- Señal de control para el bucle externo ---
        # Se pone a False cuando el usuario cierra la ventana o pulsa ESC
        self.running = True
 
        # --- Espacios de acción y observación (requerido por Gymnasium) ---
        self.action_space = spaces.Discrete(3)
 
        # Vector de 14 valores: ver _get_observation() para el detalle
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(14,), dtype=np.float32
        )
 
        # --- Estado interno (se inicializa en reset) ---
        self.snake     = None
        self.direction = None
        self.good_food = None
        self.bad_food  = None
        self.obstacles = None
        self.steps     = 0
 
        # --- Recursos de Pygame (se crean en el primer render) ---
        self.window = None
        self.clock  = None
 
    # ------------------------------------------------------------------
    # reset(): inicia un nuevo episodio
    # ------------------------------------------------------------------
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
 
        mid = self.grid_size // 2
 
        # Serpiente inicial: 3 celdas en el centro mirando a la derecha
        self.snake = deque([
            (mid, mid),
            (mid, mid - 1),
            (mid, mid - 2),
        ])
        self.direction = 1  # Derecha
        self.steps     = 0
 
        self._place_food()
        self._place_obstacles()
 
        return self._get_observation(), {}
 
    # ------------------------------------------------------------------
    # step(): nucleo del MDP — aplica la accion y devuelve la transicion
    # ------------------------------------------------------------------
    def step(self, action):
        self.steps += 1
 
        # 1. Actualizar direccion (acciones relativas a la direccion actual)
        if action == 0:                          # girar a la izquierda
            self.direction = (self.direction - 1) % 4
        elif action == 2:                        # girar a la derecha
            self.direction = (self.direction + 1) % 4
        # action == 1: seguir recto, la direccion no cambia
 
        # 2. Calcular posicion de la nueva cabeza
        dr, dc   = self.DIRECTIONS[self.direction]
        hr, hc   = self.snake[0]
        new_head = (hr + dr, hc + dc)
 
        # 3. Recompensa base por paso (penaliza pasividad)
        reward     = -0.01
        terminated = False
        info       = {"cause": None}
 
        # 4. Comprobar colision con pared o cuerpo → fin de episodio
        if self._is_collision(new_head):
            reward     = -20.0
            terminated = True
            info["cause"] = "collision"
            return self._get_observation(), reward, terminated, False, info
 
        # 5. Mover la serpiente (anadir cabeza, quitar cola si no come)
        self.snake.appendleft(new_head)
 
        if new_head == self.good_food:
            # Come comida buena: crece (no quitamos cola)
            reward += 10.0
            self._place_food()
 
        elif new_head == self.bad_food:
            # Come comida mala: fin de episodio
            reward     = -10.0
            terminated = True
            info["cause"] = "bad_food"
            return self._get_observation(), reward, terminated, False, info
 
        else:
            # Movimiento normal: eliminar cola para mantener la longitud
            self.snake.pop()
            # Reward shaping: acercarse a la comida buena da una senal positiva
            reward += self._distance_reward(new_head)
 
        # 6. Comprobar colision con obstaculo tras el movimiento
        if new_head in self.obstacles:
            reward     = -20.0
            terminated = True
            info["cause"] = "obstacle"
            return self._get_observation(), reward, terminated, False, info
 
        # 7. Mover obstaculos periodicamente (introduce estocasticidad)
        if self.steps % self.obstacle_move_freq == 0:
            self._move_obstacles()
 
        # 8. Truncamiento: limite maximo de pasos por episodio
        truncated = self.steps >= self.max_steps
 
        return self._get_observation(), reward, terminated, truncated, info
 
    # ------------------------------------------------------------------
    # render(): visualizacion del entorno con Pygame
    # ------------------------------------------------------------------
    def render(self):
        if self.render_mode != "human":
            return
 
        # Inicializar Pygame la primera vez que se llama a render()
        if self.window is None:
            pygame.init()
            pygame.display.set_caption("Snake RL — ESC para salir")
            w = self.grid_size * self.cell_size
            self.window = pygame.display.set_mode((w, w + 60))
            self.clock  = pygame.time.Clock()
 
        # --- Paleta de colores ---
        BLACK    = ( 15,  15,  20)
        DK_GRID  = ( 30,  30,  40)
        GREEN    = ( 80, 200, 100)
        RED      = (220,  60,  60)
        YELLOW   = (240, 200,  50)
        GRAY     = (100, 100, 120)
        WHITE    = (220, 220, 230)
        PANEL_BG = ( 25,  25,  35)
 
        cs = self.cell_size
 
        # Fondo
        self.window.fill(BLACK)
 
        # Cuadricula (lineas sutiles)
        for i in range(self.grid_size + 1):
            pygame.draw.line(self.window, DK_GRID,
                             (i * cs, 0), (i * cs, self.grid_size * cs))
            pygame.draw.line(self.window, DK_GRID,
                             (0, i * cs), (self.grid_size * cs, i * cs))
 
        # Obstaculos (rectangulos grises redondeados)
        for (r, c) in self.obstacles:
            rect = pygame.Rect(c*cs + 2, r*cs + 2, cs - 4, cs - 4)
            pygame.draw.rect(self.window, GRAY, rect, border_radius=4)
 
        # Comida mala (cuadrado rojo)
        if self.bad_food:
            r, c = self.bad_food
            rect = pygame.Rect(c*cs + 4, r*cs + 4, cs - 8, cs - 8)
            pygame.draw.rect(self.window, RED, rect, border_radius=cs // 4)
 
        # Comida buena (circulo amarillo)
        if self.good_food:
            r, c   = self.good_food
            cx, cy = c*cs + cs // 2, r*cs + cs // 2
            pygame.draw.circle(self.window, YELLOW, (cx, cy), cs // 2 - 4)
 
        # Cuerpo de la serpiente (degradado de verde a oscuro segun distancia a cabeza)
        for i, (r, c) in enumerate(list(self.snake)[1:], 1):
            shade = max(30, 200 - i * 8)
            rect  = pygame.Rect(c*cs + 3, r*cs + 3, cs - 6, cs - 6)
            pygame.draw.rect(self.window, (0, shade, 50), rect, border_radius=5)
 
        # Cabeza (verde brillante, mas prominente)
        hr, hc    = self.snake[0]
        head_rect = pygame.Rect(hc*cs + 1, hr*cs + 1, cs - 2, cs - 2)
        pygame.draw.rect(self.window, GREEN, head_rect, border_radius=8)
 
        # Panel inferior con metricas
        panel_y = self.grid_size * cs
        pygame.draw.rect(self.window, PANEL_BG,
                         (0, panel_y, self.grid_size * cs, 60))
        font = pygame.font.SysFont("monospace", 17)
        txt  = font.render(
            f"Longitud: {len(self.snake)}   "
            f"Pasos: {self.steps}   "
            f"Obstaculos: {len(self.obstacles)}   "
            f"[ESC] Salir",
            True, WHITE
        )
        self.window.blit(txt, (10, panel_y + 20))
 
        pygame.display.flip()
        self.clock.tick(self.metadata["render_fps"])
 
        # --- Gestion de eventos ---
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                self.close()
                return
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_ESCAPE:
                    self.close()
                    return
 
    # ------------------------------------------------------------------
    # close(): libera los recursos de Pygame
    # ------------------------------------------------------------------
    def close(self):
        self.running = False
        if self.window is not None:
            pygame.quit()
            self.window = None
 
    # ==================================================================
    # Metodos auxiliares privados
    # ==================================================================
 
    def _get_observation(self) -> np.ndarray:
        """
        Construye el vector de estado de 14 elementos.
 
        Los peligros y la posicion relativa de la comida se expresan
        en coordenadas RELATIVAS a la direccion actual del agente.
        Esto garantiza la propiedad de Markov y reduce el espacio efectivo.
 
        Indice | Descripcion
        -------|--------------------------------------------------
        [0]    | Peligro delante (pared o cuerpo u obstaculo)
        [1]    | Peligro a la izquierda
        [2]    | Peligro a la derecha
        [3-6]  | Direccion actual en one-hot (arriba/der/abajo/izq)
        [7]    | Comida buena al norte
        [8]    | Comida buena al este
        [9]    | Comida buena al sur
        [10]   | Comida buena al oeste
        [11]   | Comida mala justo delante
        [12]   | Obstaculo justo delante
        [13]   | Longitud normalizada (len / grid^2)
        """
        head    = self.snake[0]
        dir_idx = self.direction
 
        # Vectores de direccion relativos al agente
        front = self.DIRECTIONS[dir_idx]
        left  = self.DIRECTIONS[(dir_idx - 1) % 4]
        right = self.DIRECTIONS[(dir_idx + 1) % 4]
 
        def next_cell(pos, d):
            return (pos[0] + d[0], pos[1] + d[1])
 
        # [0-2] Peligro en las tres direcciones relativas
        danger_front = float(self._is_collision(next_cell(head, front)) or
                             next_cell(head, front) in self.obstacles)
        danger_left  = float(self._is_collision(next_cell(head, left))  or
                             next_cell(head, left)  in self.obstacles)
        danger_right = float(self._is_collision(next_cell(head, right)) or
                             next_cell(head, right) in self.obstacles)
 
        # [3-6] Direccion actual en codificacion one-hot
        dir_one_hot = [float(dir_idx == i) for i in range(4)]
 
        # [7-10] Posicion relativa de la comida buena (N/E/S/O)
        gf_r, gf_c = self.good_food
        h_r,  h_c  = head
        food_up    = float(gf_r < h_r)
        food_right = float(gf_c > h_c)
        food_down  = float(gf_r > h_r)
        food_left  = float(gf_c < h_c)
 
        # [11] Comida mala en la celda justo delante
        bad_front = float(next_cell(head, front) == self.bad_food)
 
        # [12] Obstaculo en la celda justo delante
        obs_front = float(next_cell(head, front) in self.obstacles)
 
        # [13] Longitud de la serpiente normalizada al tamano de la cuadricula
        length_norm = len(self.snake) / (self.grid_size ** 2)
 
        return np.array([
            danger_front, danger_left, danger_right,
            *dir_one_hot,
            food_up, food_right, food_down, food_left,
            bad_front, obs_front,
            length_norm,
        ], dtype=np.float32)
 
    def _is_collision(self, pos) -> bool:
        """True si la posicion esta fuera de la cuadricula o es parte del cuerpo."""
        r, c = pos
        if r < 0 or r >= self.grid_size or c < 0 or c >= self.grid_size:
            return True
        return pos in list(self.snake)
 
    def _distance_reward(self, head) -> float:
        """
        Reward shaping basado en distancia Manhattan a la comida buena.
        Devuelve +0.1 si el agente se acerco, -0.1 si se alejo.
        """
        gf_r, gf_c = self.good_food
        h_r,  h_c  = head
        dist_actual = abs(gf_r - h_r) + abs(gf_c - h_c)
 
        # Posicion previa: segundo elemento del deque (indice 1)
        prev_r, prev_c = list(self.snake)[1] if len(self.snake) > 1 else head
        dist_previa    = abs(gf_r - prev_r) + abs(gf_c - prev_c)
 
        if dist_actual < dist_previa:
            return  0.1
        if dist_actual > dist_previa:
            return -0.1
        return 0.0
 
    def _free_cells(self) -> list:
        """Devuelve todas las celdas que no estan ocupadas por ningun elemento."""
        occupied  = set(self.snake)
        if self.good_food:
            occupied.add(self.good_food)
        if self.bad_food:
            occupied.add(self.bad_food)
        if self.obstacles:
            occupied.update(self.obstacles)
        all_cells = {
            (r, c)
            for r in range(self.grid_size)
            for c in range(self.grid_size)
        }
        return list(all_cells - occupied)
 
    def _place_food(self):
        """Coloca comida buena y comida mala en celdas libres aleatorias."""
        free = self._free_cells()
        if len(free) >= 2:
            pos            = random.sample(free, 2)
            self.good_food = pos[0]
            self.bad_food  = pos[1]
        elif len(free) == 1:
            self.good_food = free[0]
            self.bad_food  = None
 
    def _place_obstacles(self):
        """Coloca los obstaculos en celdas libres al inicio del episodio."""
        free           = self._free_cells()
        n              = min(self.n_obstacles, len(free))
        self.obstacles = set(random.sample(free, n))
 
    def _move_obstacles(self):
        """
        Mueve cada obstaculo a una celda adyacente libre aleatoriamente.
        Introduce estocasticidad en la dinamica de transicion P(s'|s,a).
        """
        new_obstacles = set()
        for obs in self.obstacles:
            candidates = []
            for dr, dc in self.DIRECTIONS:
                nr, nc    = obs[0] + dr, obs[1] + dc
                candidate = (nr, nc)
                if (0 <= nr < self.grid_size and
                        0 <= nc < self.grid_size and
                        candidate not in self.snake and
                        candidate != self.good_food and
                        candidate != self.bad_food and
                        candidate not in new_obstacles):
                    candidates.append(candidate)
            new_obstacles.add(random.choice(candidates) if candidates else obs)
        self.obstacles = new_obstacles

In [3]:
if __name__ == "__main__":
    env = SnakeEnv(grid_size=10, render_mode="human", cell_size=50)
    obs, info = env.reset()
 
    total_reward = 0.0
    episodio     = 1
 
    print("Iniciando agente aleatorio. Pulsa ESC o cierra la ventana para salir.")
    print(f"{'Episodio':>9} | {'Recompensa':>10} | {'Longitud':>8} | Causa fin")
    print("-" * 55)
 
    while env.running:
        # Politica aleatoria: elige una accion al azar entre [0, 1, 2]
        action = env.action_space.sample()
 
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
 
        env.render()
 
        if terminated or truncated:
            causa = info.get("cause") or "max_steps"
            print(f"{episodio:>9} | {total_reward:>10.2f} | {len(env.snake):>8} | {causa}")
            episodio    += 1
            total_reward = 0.0
            obs, info    = env.reset()
 
    env.close()
    print("\nEntorno cerrado correctamente.")

Iniciando agente aleatorio. Pulsa ESC o cierra la ventana para salir.
 Episodio | Recompensa | Longitud | Causa fin
-------------------------------------------------------
        1 |     -20.00 |        3 | obstacle
        2 |     -19.97 |        3 | collision
        3 |     -20.29 |        3 | collision
        4 |     -20.39 |        3 | obstacle

Entorno cerrado correctamente.
